In [1]:
import os, time, yaml
from pyspark.sql import SparkSession, Window, functions as F

with open("/home/jovyan/work/config.yaml") as f:
    cfg = yaml.safe_load(f)

HDFS = cfg["hdfs_base"]                              # hdfs://namenode:9000
RAW = HDFS + cfg["paths"]["raw"]                     # zona raw: el CSV tal cual
CURATED = HDFS + cfg["paths"]["curated"]             # zona curada: Parquet limpio
LANG_INFO = HDFS + cfg["paths"]["language_info"]     # tabla auxiliar de idiomas
RESULTS = HDFS + cfg["paths"]["results"]             # zona de resultados

# Igual que en el Tema 5: actuamos como root en HDFS, ANTES de crear la sesión
os.environ["HADOOP_USER_NAME"] = "root"

spark = (SparkSession.builder
         .appName("wikirank-experiments")
         .master(cfg["spark"]["master"])                             # en clase: local[*]
         .config("spark.driver.host", cfg["spark"]["driver_host"])
         .config("spark.executor.memory", cfg["spark"]["executor_memory"])
         .config("spark.hadoop.dfs.replication", cfg["spark"]["dfs_replication"])
         .getOrCreate())
spark


In [2]:
articles = spark.read.parquet(CURATED)
lang_info = spark.read.parquet(LANG_INFO)
schema = ("Wikipedia_article_URL STRING, Language STRING, Page_ID LONG, "
          "Title STRING, WikiRank_score DOUBLE, WikiRank_URL STRING")
raw = spark.read.schema(schema).option("header", True).option("escape", '"').csv(RAW)

In [4]:
import csv
import time

lines = spark.sparkContext.textFile(RAW)
header = lines.first()

def parse(line):
    # Como l.split(","), pero respetando las comas que hay dentro de algunos títulos
    f = next(csv.reader([line]))
    try:
        lang, score = f[1].strip().lower(), float(f[4])
    except (IndexError, ValueError):
        return []                                        # fila rota: se descarta
    return [(lang, score)] if lang and 0 <= score <= 100 else []

t = time.time()
result = (lines.filter(lambda l: l != header)            # quitar la cabecera
               .flatMap(parse)                           # MAP: (idioma, score)
               .mapValues(lambda s: (s, 1))              # (idioma, (score, 1))
               .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))   # SHUFFLE + REDUCE
               .mapValues(lambda x: (x[1], round(x[0] / x[1], 2)))    # (idioma, (n, media))
               .sortBy(lambda kv: -kv[1][0])
               .collect())
print(f"RDD: {time.time() - t:.1f} s")
for lang, (n, avg) in result[:10]:
    print(lang, n, avg)


RDD: 327.0 s
en 6860430 24.71
de 2930971 18.63
fr 2626210 16.12
sv 2591505 20.02
nl 2163715 12.73
ru 1991787 18.18
es 1905093 18.77
it 1875385 13.63
pl 1624175 17.03
zh 1433026 20.31


**Optimización**

In [3]:
t = time.time()
raw.agg(F.avg("WikiRank_score")).show()
print(f"CSV:     {time.time() - t:.1f} s")

t = time.time()
articles.agg(F.avg("WikiRank_score")).show()
print(f"Parquet: {time.time() - t:.1f} s")

# ¿Por qué? En la línea FileScan, ReadSchema: Parquet solo lee la columna WikiRank_score
articles.agg(F.avg("WikiRank_score")).explain()

+-------------------+
|avg(WikiRank_score)|
+-------------------+
| 17.932223616843302|
+-------------------+

CSV:     78.0 s
+-------------------+
|avg(WikiRank_score)|
+-------------------+
| 17.932223616844187|
+-------------------+

Parquet: 4.8 s
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[], functions=[avg(WikiRank_score#2)])
   +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=92]
      +- HashAggregate(keys=[], functions=[partial_avg(WikiRank_score#2)])
         +- Project [WikiRank_score#2]
            +- FileScan parquet [WikiRank_score#2,Language#4] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[hdfs://namenode:9000/curated/articles], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<WikiRank_score:double>




In [4]:
from pyspark.sql import functions as F

# Definimos la función de la Pregunta 3 actual: une los 47M de artículos con la metadata de idiomas
def q3_expr(info_table):
    return (articles
        .withColumn("title_length", F.length("Title"))
        .withColumn("title_words", F.size(F.split(F.trim("Title"), r"\s+")))
        .join(info_table, "Language", "left")
        .groupBy("Language", "language_name", "family")
        .agg(
            F.count("*").alias("n_articles"),
            F.round(F.covar_samp("title_length", "WikiRank_score"), 2).alias("covar_title_score"),
            F.round(F.corr("title_length", "WikiRank_score"), 3).alias("corr_title_score"),
            F.round(F.corr("title_words", "WikiRank_score"), 3).alias("corr_words_score"),
            F.round(F.corr("title_length", "title_words"), 3).alias("corr_length_words")
        )
    )

# Desactivamos el broadcast automático en Spark (-1 = fuerza Sort-Merge Join)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# Seleccionamos las columnas de la tabla auxiliar
lang_sub = lang_info.select("Language", "language_name", "family")

q_smj = q3_expr(lang_sub)
q_smj.explain()  # Muestra el plan de ejecución: verás los "Exchange" (shuffle por la red)

t = time.time()
q_smj.write.format("noop").mode("overwrite").save()  # noop: ejecuta todo el cálculo sin guardar en disco
print(f"Sort-merge join (sin broadcast): {time.time() - t:.1f} s")


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Language#4, language_name#6, family#9], functions=[count(1), covar_samp(cast(title_length#71 as double), WikiRank_score#2), corr(cast(title_length#71 as double), WikiRank_score#2), corr(cast(title_words#72 as double), WikiRank_score#2), corr(cast(title_length#71 as double), cast(title_words#72 as double))])
   +- HashAggregate(keys=[Language#4, language_name#6, family#9], functions=[partial_count(1), partial_covar_samp(cast(title_length#71 as double), WikiRank_score#2), partial_corr(cast(title_length#71 as double), WikiRank_score#2), partial_corr(cast(title_words#72 as double), WikiRank_score#2), partial_corr(cast(title_length#71 as double), cast(title_words#72 as double))])
      +- Project [Language#4, WikiRank_score#2, title_length#71, title_words#72, language_name#6, family#9]
         +- SortMergeJoin [Language#4], [Language#5], LeftOuter
            :- Sort [Language#4 ASC NULLS FIRST], false, 0
      

In [ ]:
# Restauramos el umbral por defecto (10 MB)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)

# Forzamos explícitamente el broadcast de la tabla auxiliar pequeña
q_bc = q3_expr(F.broadcast(lang_sub))
q_bc.explain()  # Muestra el plan de ejecución: verás "BroadcastExchange" y "BroadcastHashJoin"

t = time.time()
q_bc.write.format("noop").mode("overwrite").save()
print(f"Broadcast join: {time.time() - t:.1f} s")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Language#4, language_name#6, family#9], functions=[count(1), covar_samp(cast(title_length#270 as double), WikiRank_score#2), corr(cast(title_length#270 as double), WikiRank_score#2), corr(cast(title_words#271 as double), WikiRank_score#2), corr(cast(title_length#270 as double), cast(title_words#271 as double))])
   +- Exchange hashpartitioning(Language#4, language_name#6, family#9, 200), ENSURE_REQUIREMENTS, [plan_id=333]
      +- HashAggregate(keys=[Language#4, language_name#6, family#9], functions=[partial_count(1), partial_covar_samp(cast(title_length#270 as double), WikiRank_score#2), partial_corr(cast(title_length#270 as double), WikiRank_score#2), partial_corr(cast(title_words#271 as double), WikiRank_score#2), partial_corr(cast(title_length#270 as double), cast(title_words#271 as double))])
         +- Project [Language#4, WikiRank_score#2, title_length#270, title_words#271, language_name#6, family#9]